# 02 · Análisis exploratorio

**Clase 4:** distribución univariante, boxplot, dispersión y matriz de correlación. **Clase 3:** cobertura y anomalías antes de modelar. Los gráficos usan datos observados, no datos inventados.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
A = pd.read_parquet(DATA_PROCESSED / 'dataset_A_pvod_hourly.parquet')
B = pd.read_parquet(DATA_PROCESSED / 'dataset_B_argentina_mensual.parquet')
sol = A[A['is_daylight'].eq(1)].copy()
cobertura = A.groupby('plant_id').agg(desde=('date_local','min'), hasta=('date_local','max'),
                                     horas=('pvod_power','size'), capacidad_mw=('plant_capacity_mw','first'),
                                     pico_mw=('pvod_power','max'))
print(cobertura)
print('Faltantes:', A.isna().sum().sum(), '| plantas Argentina:', B.central.nunique())


## Distribución de potencia por planta

El boxplot muestra mediana, cuartiles, bigotes y valores extremos. Un extremo es una observación para investigar: no confirma una falla. La potencia se divide por la capacidad nominal para comparar escalas.

In [ ]:
px.box(sol, x='plant_id', y='pvod_power_norm', points='outliers',
       labels={'plant_id':'Planta','pvod_power_norm':'Potencia / capacidad (MW/MW)'},
       title='Clase 4 · Distribución de potencia observada').show()
px.histogram(sol, x='pvod_power_norm', nbins=45, title='Clase 4 · Histograma de potencia normalizada').show()


## Relación entre irradiancia y generación

La dispersión y la correlación describen asociaciones. Una correlación alta no confirma causalidad ni el desempeño de un modelo en datos nuevos.

In [ ]:
muestra = sol.sample(min(2000,len(sol)),random_state=42)
px.scatter(muestra,x='pvod_lmd_totalirrad',y='pvod_power_norm',color='plant_id',opacity=.4,
           labels={'pvod_lmd_totalirrad':'Irradiancia medida (W/m²)','pvod_power_norm':'Potencia / capacidad (MW/MW)'},
           title='Clase 4 · Irradiancia y potencia').show()
cols=['pvod_power_norm','pvod_lmd_totalirrad','pvod_lmd_temperature','om_shortwave_radiation']
px.imshow(sol[cols].corr(), zmin=-1,zmax=1,color_continuous_scale='RdBu_r',text_auto='.2f',
          title='Clase 4 · Correlación entre variables observadas').show()


## Cobertura argentina y ceros observados

Los centroides provinciales se excluyen de comparaciones meteorológicas por parque. Las ubicaciones inferidas de una localidad se identifican como aproximadas. Un mes con cero posterior al inicio operativo se conserva para evaluar; no se utiliza para aprender producción sana.

In [ ]:
print(B.groupby('coord_source').central.nunique())
print('Meses con cero generación:', B.energia_mwh.eq(0).sum())
anual=B.groupby('anio',as_index=False).agg(gwh=('energia_gwh','sum'),meses=('mes','nunique'))
print(anual)
px.bar(anual,x='anio',y='gwh',hover_data=['meses'],title='CAMMESA · Energía solar anual (2026 incompleto)',
       labels={'gwh':'GWh','anio':'Año'}).show()


## Hallazgos para el modelado

- La irradiancia está asociada con la potencia, con diferencias entre plantas.
- station04 excede su capacidad declarada: se muestra sin interpretación económica.
- station09 tiene un comportamiento distinto y queda como caso de aplicación del modelo genérico.
- Hay una sola vuelta estacional; no se puede confirmar una causa física solo con estos datos.
- China admite análisis diario con sensores. Argentina tiene datos mensuales y cobertura de potencia/coordenadas desigual.
